# SLM-KDN Colab Pro A100: Semantic RAG Micro-KDN

This notebook tests the final architecture:

1. Fine-tuned Llama-3-8B emits strict semantic JSON only.
2. The local command-context/template store retrieves command metadata.
3. Python deterministically assembles Junos CLI.
4. Guardrails enforce commit and mode validity.

Use the older RAG notebook for the baseline `--use_rag` free-form command path. Use this notebook for the final `--semantic_rag` path.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))

## 2. Clone Repo From Master

In [ ]:
%cd /content
!rm -rf slm-kdn
!git clone --branch master https://github.com/Surya-Prasad/slm-kdn.git
%cd /content/slm-kdn
!git status --short
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets pypdf nltk

## 4. Hugging Face Login

Run this if your base model or adapter needs gated Hugging Face access.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Preprocess NIT Dataset

This creates `data/processed/train.jsonl`, `val.jsonl`, `test.jsonl`, and robustness variants when available.

In [ ]:
!bash scripts/run_preprocess.sh
!find data/processed -maxdepth 1 -type f -print | sort
!head -n 1 data/processed/test.jsonl

## 6. Build Command-Context Datastore

This builds `data/juniper_templates.json` from train/val only. It does not use `test.jsonl` unless `--allow-test` is explicitly passed, which you should not use for final evaluation.

In [ ]:
!python scripts/build_perfect_datastore_v2.py
!python - <<'PY'
import json
from pathlib import Path
p = Path('data/juniper_templates.json')
payload = json.loads(p.read_text())
print('template_count:', len(payload))
for i, (k, v) in enumerate(payload.items()):
    if i >= 5:
        break
    print(k, '=>', {x: v.get(x) for x in ['template', 'mode', 'requires_commit', 'allowed_params']})
PY
!wc -l outputs/datastore_conflicts.jsonl || true
!head -n 5 outputs/datastore_conflicts.jsonl || true

## 7. Run Semantic RAG Unit Smoke Tests

In [ ]:
!python tests/test_semantic_rag.py

## 8. Train LoRA Adapter

Run this if your current adapter has not been trained in this runtime. For the semantic architecture, the important thing to inspect is the semantic parser metrics below: if JSON validity is low, the adapter is still behaving like the old command-generation model and needs semantic-JSON supervised examples.

In [ ]:
!bash scripts/run_train.sh

## 9. Semantic RAG Smoke Inference

This verifies that `--semantic_rag` writes semantic JSON, command context, assembly errors, commit decisions, and guardrail diagnostics.

In [ ]:
import json, os
os.makedirs('data/processed', exist_ok=True)
rows = [
    {'intent': 'Display chassis LED status', 'context': '', 'target_command': 'show chassis led', 'category': 'semantic_smoke'},
    {'intent': 'Clear all MAC address entries in the ethernet switching table', 'context': '', 'target_command': 'clear ethernet-switching-table', 'category': 'semantic_smoke'},
    {'intent': 'Disable the OSPF protocol', 'context': '', 'target_command': 'set protocols ospf disable\\ncommit', 'category': 'semantic_smoke'},
]
with open('data/processed/semantic_rag_smoke.jsonl', 'w', encoding='utf-8') as f:
    for row in rows:
        f.write(json.dumps(row) + '\n')

!mkdir -p results/predictions results/metrics results/error_analysis
!python src/infer.py \
  --input_file data/processed/semantic_rag_smoke.jsonl \
  --output_file results/predictions/semantic_rag_smoke_predictions.jsonl \
  --semantic_rag \
  --mode intent_with_context

!python scripts/evaluate_semantic_rag.py \
  --pred_file results/predictions/semantic_rag_smoke_predictions.jsonl \
  --out_file results/metrics/semantic_rag_smoke_metrics.json \
  --failures_file results/error_analysis/semantic_rag_smoke_failures.jsonl \
  --summary_file results/error_analysis/semantic_rag_smoke_error_summary.json

!head -n 3 results/predictions/semantic_rag_smoke_predictions.jsonl
!cat results/metrics/semantic_rag_smoke_metrics.json

## 10. Full Semantic RAG Evaluation

This runs `test.jsonl` plus `clean_test.jsonl`, `paraphrased_test.jsonl`, and `noisy_test.jsonl` if present. Metrics are written under `results/metrics/`; stage failures and summaries are written under `results/error_analysis/`.

In [ ]:
!scripts/run_semantic_rag_eval.sh
!cat results/metrics/semantic_rag_metrics.json
!cat results/error_analysis/semantic_rag_error_summary.json
!head -n 5 results/error_analysis/semantic_rag_failures.jsonl || true

## 11. Compare Against Legacy RAG Baseline Optional

Use this only for comparison. The final architecture is `--semantic_rag` above.

In [ ]:
!python src/infer.py \
  --input_file data/processed/test.jsonl \
  --output_file results/predictions/rag_predictions_baseline.jsonl \
  --use_rag \
  --rag-corpus train,rag_docs
!python src/evaluate.py \
  --pred_file results/predictions/rag_predictions_baseline.jsonl \
  --out_file results/metrics/rag_baseline_eval_metrics.json
!cat results/metrics/rag_baseline_eval_metrics.json

## 12. Save Results To Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/slm-kdn-results
!cp -r results /content/drive/MyDrive/slm-kdn-results/semantic-rag-results-$(date +%Y%m%d-%H%M%S)
!cp data/juniper_templates.json /content/drive/MyDrive/slm-kdn-results/juniper_templates-$(date +%Y%m%d-%H%M%S).json
!ls -lh /content/drive/MyDrive/slm-kdn-results